# Case study: a leaking product-rating feature

*Session 9, block 2 practice. Author: course team, licence CC-BY-4.0.*

`products.parquet` contains `train_avg_rating`, the mean rating of each product over **all** training reviews. For a training review this mean includes the review's own rating and later ratings. In this notebook we

1. show that the feature leaks the label,
2. replace it with the mean of **earlier** reviews of the same product,
3. compare both versions with a validation that imitates the leaderboard.

Theory: [02-aggregates-leakage-and-text-statistics.md](../theory/02-aggregates-leakage-and-text-statistics.md).

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# find the repository root, wherever the notebook is started from
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "case-study" / "data").exists())
DATA = ROOT / "case-study" / "data"

reviews = pd.read_parquet(DATA / "train_sample.parquet")
products = pd.read_parquet(DATA / "products.parquet", columns=["parent_asin", "train_avg_rating", "train_n_reviews"])
df = reviews.merge(products, on="parent_asin", how="left")
df[["rating", "train_avg_rating", "train_n_reviews"]].describe().round(2)

## 1. Detect the leak

Three checks:

- products with a single training review: the feature equals the label,
- the correlation of the feature with the rating,
- the mean of the feature per label.

In [ ]:
single = df["train_n_reviews"] == 1
print("share of reviews whose product has one training review:", round(single.mean(), 3))
print("feature equals the rating for these reviews:", (df.loc[single, "train_avg_rating"] == df.loc[single, "rating"]).mean())
print("correlation with the rating:", round(df["rating"].corr(df["train_avg_rating"]), 3))
df.groupby("label")["train_avg_rating"].mean().round(2)

**Task 1.** Plot the mean of `train_avg_rating` per star rating (1–5) for products with 2, 3–5, 6–20 and more than 20 reviews (`pd.cut` on `train_n_reviews`). Where is the leak strongest, and why?

In [ ]:
# TODO (Task 1)


## 2. Replace it with the mean of earlier reviews

The sample has only a few reviews per product, so the history is computed on the full training table and joined by `review_id`. Sorting by date before the grouped cumulative sum is essential.

In [ ]:
# product history from the FULL training table, in time order
train = pd.read_parquet(DATA / "train.parquet", columns=["review_id", "parent_asin", "user_id", "rating", "date"])
train = train.sort_values(["date", "review_id"]).reset_index(drop=True)
g = train.groupby("parent_asin")["rating"]
train["prod_prev_n"] = g.cumcount()
train["prod_prev_mean"] = ((g.cumsum() - train["rating"]) / train["prod_prev_n"]).where(train["prod_prev_n"] > 0)
train["user_prev_n"] = train.groupby("user_id").cumcount()
history = train[["review_id", "prod_prev_n", "prod_prev_mean", "user_prev_n"]]

df = df.merge(history, on="review_id", how="left")
print("first reviews of a product (no history):", round(df["prod_prev_mean"].isna().mean(), 3))
print("correlation with the rating:", round(df["rating"].corr(df["prod_prev_mean"]), 3))
df.groupby("label")[["train_avg_rating", "prod_prev_mean"]].mean().round(2)

**Task 2.** Check the computation by hand for one product: pick a `parent_asin` with 4–6 reviews in `train`, print its reviews in time order and verify `prod_prev_mean` for each row.

In [ ]:
# TODO (Task 2)
example = train["parent_asin"].value_counts().loc[lambda s: s.between(4, 6)].index[0]
train.loc[train["parent_asin"] == example, ["date", "rating", "prod_prev_n", "prod_prev_mean"]]

## 3. Compare both versions like the leaderboard

Fit on the sample reviews up to 2020 and validate on 2021. For the 2021 rows the realistic product feature is the mean **as of 1 January 2021** (all earlier reviews), which is exactly how the test rows of the leaderboard are built from `train_avg_rating`.

In [ ]:
NEGATIONS = r"\b(?:not|no|never|don't|doesn't|didn't|isn't|wasn't|won't|can't)\b"


def text_stats(frame):
    """Seven numeric statistics of review text and title (Session 9, page 2)."""
    text, title = frame["text"].fillna(""), frame["title"].fillna("")
    n_chars = text.str.len()
    return pd.DataFrame({
        "log_chars": np.log1p(n_chars),
        "n_words": text.str.split().str.len(),
        "n_exclaim": text.str.count("!"),
        "n_question": text.str.count(r"\?"),
        "upper_share": text.str.count(r"[A-Z]") / n_chars.clip(lower=1),
        "n_negations": text.str.lower().str.count(NEGATIONS),
        "title_words": title.str.split().str.len(),
    }, index=frame.index)

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import f1_score

cut = pd.Timestamp("2021-01-01")
asof = train[train["date"] < cut].groupby("parent_asin")["rating"].mean().rename("asof_mean")
df = df.merge(asof, on="parent_asin", how="left")
base = pd.concat([text_stats(df), df[["verified_purchase", "helpful_vote", "n_images"]].astype(float)], axis=1)
fit, val = df["date"] < cut, df["date"] >= cut


def macro_f1(fit_col=None, val_col=None):
    X_fit, X_val = base[fit].copy(), base[val].copy()
    if fit_col:
        X_fit["product_rating"] = df.loc[fit, fit_col]
        X_val["product_rating"] = df.loc[val, val_col]
    model = HistGradientBoostingClassifier(class_weight="balanced", random_state=0)
    model.fit(X_fit, df.loc[fit, "label"])
    return round(f1_score(df.loc[val, "label"], model.predict(X_val), average="macro"), 3)


pd.Series({
    "no product rating": macro_f1(),
    "leaky / leaky": macro_f1("train_avg_rating", "train_avg_rating"),
    "leaky / as of 2021": macro_f1("train_avg_rating", "asof_mean"),
    "past mean / past mean": macro_f1("prod_prev_mean", "prod_prev_mean"),
    "past mean / as of 2021": macro_f1("prod_prev_mean", "asof_mean"),
}, name="macro-F1 on 2021")

**Task 3.** Write three sentences for a team member who wants to use `train_avg_rating` "because it improves the CV score by almost 0.1".

**Task 4.** Add `prod_prev_n` and `user_prev_n` as features. Does the number of earlier reviews help? For the test rows, which columns of `products.parquet` correspond to `prod_prev_mean` and `prod_prev_n`?

**Task 5 (optional).** Build a past-only share of *neutral* earlier reviews per product (`prod_prev_neu_share`) with the same cumulative-sum trick, and test it.

In [ ]:
# TODO (Tasks 4 and 5)
